In [1]:
from pathlib import Path
print(Path.cwd())

f:\Big Data\Retail_Lakehouse_Learning\ingestion


In [2]:
from pathlib import Path  # Import Path, which helps us work with folders and filenames.

project_root = Path.cwd().parent  # Get the notebook's current folder, then move up from ingestion to the project folder.

workbook_path = project_root / "Requirements" / "Retail_Lakehouse_Dummy_Data.xlsx"  # Build the path to the source workbook.

print(workbook_path)  # Show the full path so we can check that it points to the right file.

print("File found:", workbook_path.exists())  # Check whether a file exists at that path; the result is True or False.

f:\Big Data\Retail_Lakehouse_Learning\Requirements\Retail_Lakehouse_Dummy_Data.xlsx
File found: True


In [3]:
from openpyxl import load_workbook  # Import the function that opens an Excel workbook.

workbook = load_workbook(workbook_path, read_only=True, data_only=True)  # Open the workbook for reading and store it in the variable workbook.

print("Number of sheets:", len(workbook.sheetnames))  # Count the worksheet names.

print("Sheet names:", workbook.sheetnames)  # Display the names so we can identify the available datasets.

Number of sheets: 10
Sheet names: ['customers', 'orders', 'order_items', 'payments', 'products', 'stores', 'promotions', 'suppliers', 'weather_sample', 'stream_events']


In [4]:
customers_sheet = workbook["customers"]  # Select the worksheet named customers.

headers = next(customers_sheet.iter_rows(min_row=1, max_row=1, values_only=True))  # Read row 1, which contains the column names.

first_customer = next(customers_sheet.iter_rows(min_row=2, max_row=2, values_only=True))  # Read row 2, the first customer record.

print("Columns:", headers)  # Display the column names.

print("First customer:", first_customer)  # Display one example record.

print("Data rows:", customers_sheet.max_row - 1)  # Subtract the header row from the last used row number.

Columns: ('customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'loyalty_tier', 'join_date', 'active_flag')
First customer: ('C00001', 'Maryam', 'Iqbal', 'maryam.iqbal.1@example.com', '+92-333-8106245', 'Lahore', 'Gold', '2026-05-13', True)
Data rows: 750


In [5]:
for sheet_name in workbook.sheetnames:  # Repeat the indented lines once for each worksheet name.
    sheet = workbook[sheet_name]  # Open the worksheet with the current name.
    data_rows = sheet.max_row - 1  # Estimate data rows by excluding the header.
    print(f"{sheet_name}: {data_rows} rows, {sheet.max_column} columns")  # Display its name, row count, and column count.

customers: 750 rows, 9 columns
orders: 4000 rows, 13 columns
order_items: 9868 rows, 7 columns
payments: 4000 rows, 7 columns
products: 120 rows, 8 columns
stores: 6 rows, 8 columns
promotions: 15 rows, 8 columns
suppliers: 18 rows, 8 columns
weather_sample: 552 rows, 8 columns
stream_events: 100 rows, 10 columns


Validate customer IDs

In [6]:
customer_ids = []  # Make a list to hold non-blank customer IDs.
blank_customer_ids = 0  # Start a counter for missing IDs at zero.

for row in customers_sheet.iter_rows(min_row=2, values_only=True):  # Read each customer row, skipping the header.
    customer_id = row[0]  # Take the first column, which is customer_id.

    if customer_id is None or str(customer_id).strip() == "":  # Check for an empty cell or whitespace-only value.
        blank_customer_ids += 1  # Count this row as having a blank ID.
    else:  # Run this part when the ID contains a value.
        customer_ids.append(customer_id)  # Add the ID to our list.

duplicate_customer_ids = len(customer_ids) - len(set(customer_ids))  # Compare all IDs with the unique IDs in a set.

print("Blank customer IDs:", blank_customer_ids)  # Show the missing-ID count.
print("Duplicate customer IDs:", duplicate_customer_ids)  # Show the duplicate-row count.

Blank customer IDs: 0
Duplicate customer IDs: 0


In [7]:
known_customer_ids = set(customer_ids)  # Keep each valid customer ID once for quick lookup.

orders_sheet = workbook["orders"]  # Select the orders worksheet.

unknown_customer_count = 0  # Start the count of orders with no matching customer.

for row in orders_sheet.iter_rows(min_row=2, values_only=True):  # Read each order, skipping the header.
    order_customer_id = row[1]  # Column B contains the order's customer_id; Python starts counting at 0.

    if order_customer_id not in known_customer_ids:  # Check whether that ID exists among customers.
        unknown_customer_count += 1  # Count an order whose customer is missing.

print("Orders with unknown customers:", unknown_customer_count)  # Display the final count.

Orders with unknown customers: 0


Validate Order IDs

In [8]:
orders_ids = []  # Make a list to hold non-blank order IDs.
blank_order_ids = 0  # Start a counter for missing IDs at zero.

for row in orders_sheet.iter_rows(min_row=2, values_only=True):  # Read each order row, skipping the header.
    order_id = row[0]  # Take the first column, which is order_id.

    if order_id is None or str(order_id).strip() == "":  # Check for an empty cell or whitespace-only value.
        blank_order_ids += 1  # Count this row as having a blank ID.
    else:  # Run this part when the ID contains a value.
        orders_ids.append(order_id)  # Add the ID to our list.

duplicate_order_ids = len(orders_ids) - len(set(orders_ids))  # Compare all IDs with the unique IDs in a set.

print("Blank order IDs:", blank_order_ids)  # Show the missing-ID count.
print("Duplicate order IDs:", duplicate_order_ids)  # Show the duplicate-row count.



Blank order IDs: 0
Duplicate order IDs: 0


In [9]:
orders_sheet = workbook["orders"]  # Select the sheet containing valid orders.
order_items_sheet = workbook["order_items"]  # Select the sheet containing order items.

known_order_ids = set()  # Create an empty set for valid order IDs.

for row in orders_sheet.iter_rows(min_row=2, values_only=True):  # Read each order after the header.
    order_id = row[0]  # Get order_id from column A.
    if order_id is not None:  # Keep only IDs that are not blank.
        known_order_ids.add(order_id)  # Add the ID to the set.

unknown_order_count = 0  # Start a counter for items with no matching order.

for row in order_items_sheet.iter_rows(min_row=2, values_only=True):  # Read each item after the header.
    item_order_id = row[1]  # Get the item's order_id from column B.
    if item_order_id not in known_order_ids:  # Check whether its parent order exists.
        unknown_order_count += 1  # Count this item if its order is missing.

print("Order items with unknown orders:", unknown_order_count)  # Show the result.

Order items with unknown orders: 0


Validate Product IDs

In [12]:
product_sheet = workbook["products"]  # Select the products worksheet.

product_ids = []  # Store every non-blank product ID.
blank_product_ids = 0  # Count rows with a blank product ID.

for row in product_sheet.iter_rows(min_row=2, values_only=True):  # Read product rows after the header.
    product_id = row[0]  # Get product_id from column A.

    if product_id is None or str(product_id).strip() == "":  # Check for an empty or whitespace-only ID.
        blank_product_ids += 1  # Count the blank ID.
    else:  # Run this part when the ID contains a value.
        product_ids.append(product_id)  # Add the ID to the list.

duplicate_product_ids = len(product_ids) - len(set(product_ids))  # Count IDs repeated beyond their first occurrence.

print("Non-blank product IDs:", len(product_ids))  # Confirm that we read the 120 products.
print("Blank product IDs:", blank_product_ids)  # Show missing IDs.
print("Duplicate product IDs:", duplicate_product_ids)  # Show repeated IDs.

Non-blank product IDs: 120
Blank product IDs: 0
Duplicate product IDs: 0


In [13]:
order_items_sheet = workbook["order_items"]  # Select the order-items worksheet.

known_product_ids = set(product_ids)  # Make a set of valid product IDs from the previous cell.
unknown_product_count = 0  # Count items that reference a missing product.

for row in order_items_sheet.iter_rows(min_row=2, values_only=True):  # Read item rows after the header.
    item_product_id = row[2]  # Get product_id from column C; Python calls this position 2.

    if item_product_id not in known_product_ids:  # Check whether the product exists.
        unknown_product_count += 1  # Count the item if its product is missing.

print("Known products:", len(known_product_ids))  # Confirm the lookup set contains 120 products.
print("Order items with unknown products:", unknown_product_count)  # Show broken product references.

Known products: 120
Order items with unknown products: 0


In [14]:
import csv  # Import Python's CSV-writing tools.

postgres_seed_dir = project_root / "landing" / "postgres_seed"  # Choose the folder for database seed files.
postgres_seed_dir.mkdir(parents=True, exist_ok=True)  # Create that folder if it does not exist.

customers_csv_path = postgres_seed_dir / "customers.csv"  # Choose the output filename.

with customers_csv_path.open("w", newline="", encoding="utf-8") as output_file:  # Open the CSV for writing; close it automatically afterward.
    writer = csv.writer(output_file)  # Create an object that writes correctly formatted CSV rows.
    for row in customers_sheet.iter_rows(values_only=True):  # Read every workbook row, including the header.
        writer.writerow(row)  # Write that row to the CSV.

print("Created:", customers_csv_path)  # Show where the file was saved.

Created: f:\Big Data\Retail_Lakehouse_Learning\landing\postgres_seed\customers.csv


In [15]:
with customers_csv_path.open("r", newline="", encoding="utf-8") as input_file:  # Open the generated CSV for reading.
    reader = csv.reader(input_file)  # Read it as CSV records, including the header.
    csv_headers = next(reader)  # Take the first record as the column names.
    csv_rows = list(reader)  # Read the remaining records into a list.

print("Headers match:", tuple(csv_headers) == headers)  # Compare CSV headers with the workbook headers.
print("Data rows:", len(csv_rows))  # Count exported customer records.
print("First customer ID:", csv_rows[0][0])  # Check the first exported ID.

Headers match: True
Data rows: 750
First customer ID: C00001


In [17]:
postgres_sheet_names = ["orders", "order_items", "payments"]  # Choose the three remaining PostgreSQL seed sheets.

for sheet_name in postgres_sheet_names:  # Repeat the indented block for each sheet.
    sheet = workbook[sheet_name]  # Get the current worksheet.
    output_path = postgres_seed_dir / f"{sheet_name}.csv"  # Choose its CSV filename.
    rows_written = 0  # Start counting all written rows, including the header.

    with output_path.open("w", newline="", encoding="utf-8") as output_file:  # Open the CSV and close it automatically afterward.
        writer = csv.writer(output_file)  # Create the CSV writer.

        for row in sheet.iter_rows(values_only=True):  # Read every worksheet row, including the header.
            writer.writerow(row)  # Write that row to the CSV.
            rows_written += 1  # Count the row just written.

    print(f"{sheet_name}: {rows_written - 1} data rows; file exists: {output_path.exists()}")  # Report data rows and confirm the file exists.

orders: 4000 data rows; file exists: True
order_items: 9868 data rows; file exists: True
payments: 4000 data rows; file exists: True


In [18]:
for sheet_name in ["customers", "orders", "order_items", "payments"]:  # Check each PostgreSQL seed file.
    sheet = workbook[sheet_name]  # Get its original worksheet.
    expected_header = next(sheet.iter_rows(min_row=1, max_row=1, values_only=True))  # Read the workbook header.
    expected_rows = sheet.max_row - 1  # Get the expected number of data rows.
    csv_path = postgres_seed_dir / f"{sheet_name}.csv"  # Find the exported CSV.

    with csv_path.open("r", newline="", encoding="utf-8") as input_file:  # Open the saved file for reading.
        reader = csv.reader(input_file)  # Read CSV records correctly.
        actual_header = next(reader)  # Read its header.
        actual_rows = sum(1 for _ in reader)  # Count the remaining CSV records.

    print(f"{sheet_name}: headers match={tuple(actual_header) == expected_header}, rows={actual_rows}/{expected_rows}")  # Compare export with source.

customers: headers match=True, rows=750/750
orders: headers match=True, rows=4000/4000
order_items: headers match=True, rows=9868/9868
payments: headers match=True, rows=4000/4000


In [19]:
flat_file_dir = project_root / "landing" / "flat_files"  # Choose a separate folder for CSV source files.
flat_file_dir.mkdir(parents=True, exist_ok=True)  # Create the folder if needed.

flat_sheet_names = ["products", "stores", "promotions", "suppliers"]  # Name the four flat-file sources.

for sheet_name in flat_sheet_names:  # Export one sheet at a time.
    sheet = workbook[sheet_name]  # Get the current worksheet.
    output_path = flat_file_dir / f"{sheet_name}.csv"  # Give the CSV the sheet's name.
    rows_written = 0  # Count rows written, including the header.

    with output_path.open("w", newline="", encoding="utf-8") as output_file:  # Open the output file.
        writer = csv.writer(output_file)  # Create a CSV writer.

        for row in sheet.iter_rows(values_only=True):  # Read every row from the worksheet.
            writer.writerow(row)  # Write the row to the CSV.
            rows_written += 1  # Count it.

    print(f"{sheet_name}: {rows_written - 1} data rows; file exists: {output_path.exists()}")  # Report the result.

products: 120 data rows; file exists: True
stores: 6 data rows; file exists: True
promotions: 15 data rows; file exists: True
suppliers: 18 data rows; file exists: True


In [20]:
for sheet_name in flat_sheet_names:  # Check products, stores, promotions, and suppliers one at a time.
    sheet = workbook[sheet_name]  # Get the original worksheet for this dataset.
    expected_header = next(sheet.iter_rows(min_row=1, max_row=1, values_only=True))  # Read its column names.
    expected_rows = sheet.max_row - 1  # Count its data rows, excluding the header.
    csv_path = flat_file_dir / f"{sheet_name}.csv"  # Find the CSV we created for this sheet.

    with csv_path.open("r", newline="", encoding="utf-8") as input_file:  # Open the CSV for reading.
        reader = csv.reader(input_file)  # Read its records as CSV.
        actual_header = next(reader)  # Read the CSV column names.
        actual_rows = sum(1 for _ in reader)  # Count the CSV data records.

    print(f"{sheet_name}: headers match={tuple(actual_header) == expected_header}, rows={actual_rows}/{expected_rows}")  # Compare CSV with workbook.

products: headers match=True, rows=120/120
stores: headers match=True, rows=6/6
promotions: headers match=True, rows=15/15
suppliers: headers match=True, rows=18/18


In [21]:
weather_sheet = workbook["weather_sample"]  # Select the supplied weather sample worksheet.

weather_dir = project_root / "landing" / "weather_fallback"  # Choose a separate folder for fallback data.
weather_dir.mkdir(parents=True, exist_ok=True)  # Create the folder if needed.

weather_csv_path = weather_dir / "weather_sample.csv"  # Choose the output filename.

with weather_csv_path.open("w", newline="", encoding="utf-8") as output_file:  # Open the CSV for writing.
    writer = csv.writer(output_file)  # Create a CSV writer.
    for row in weather_sheet.iter_rows(values_only=True):  # Read every sheet row, including the header.
        writer.writerow(row)  # Write the row to the CSV.

expected_header = next(weather_sheet.iter_rows(min_row=1, max_row=1, values_only=True))  # Get the original header.
expected_rows = weather_sheet.max_row - 1  # Get the original data-row count.

with weather_csv_path.open("r", newline="", encoding="utf-8") as input_file:  # Reopen the saved CSV.
    reader = csv.reader(input_file)  # Read CSV records.
    actual_header = next(reader)  # Read the saved header.
    actual_rows = sum(1 for _ in reader)  # Count the saved data records.

print(f"Weather fallback: headers match={tuple(actual_header) == expected_header}, rows={actual_rows}/{expected_rows}")  # Show both checks.

Weather fallback: headers match=True, rows=552/552
